In [ ]:
import os, sys, json, random, subprocess, importlib.util
from pathlib import Path
import numpy as np
import torch

IS_KAGGLE = Path('/kaggle').exists()
WORK_DIR = Path('/kaggle/working' if IS_KAGGLE else '/mnt/data/sigk_v4_local').resolve()
INPUT_DIR = Path('/kaggle/input' if IS_KAGGLE else WORK_DIR/'input')
OUTPUT_DIR = WORK_DIR/'sigk_v4_results'
WORK_DIR.mkdir(parents=True,exist_ok=True); OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
os.chdir(WORK_DIR)
Path('sigk_v4').mkdir(exist_ok=True)
Path('sigk_v4/__init__.py').write_text('"""SIGK V4 notebook package."""\n',encoding='utf8')
SMOKE_TEST = os.environ.get('SIGK_V4_SMOKE','0') == '1'


RUN_VARIANTS = ['drunet_lite','rescross','drunet_lite_x4','rescross_x4']
EPOCHS = 14
BATCH_SIZE = 32
SAMPLES_PER_EPOCH = 4800
PATCH_SIZE = 256
LEARNING_RATE = 2e-4
MIN_LEARNING_RATE = 1e-6
SSIM_WEIGHT = 1.0
EMA_DECAY = 0.999
NUM_WORKERS = 4
USE_AMP = True
MULTI_GPU = True
DRUNET_WIDTH = 32
RESCROSS_WIDTH = 64
SEED = 42
EVAL_SEED = 123
AUTO_DOWNLOAD_IF_MISSING = True
RUN_TRAINING = True
RUN_BASELINE_TUNING_IF_MISSING = True
FORCE_RETUNE_BASELINE = False
RUN_EVALUATION = True
RUN_LPIPS = True
EXTRA_SIGMA = None


V2_CHECKPOINT_PATH = None
V3_CHECKPOINT_PATH = None

if SMOKE_TEST:
    EPOCHS, BATCH_SIZE, SAMPLES_PER_EPOCH, PATCH_SIZE = 1, 2, 4, 32
    DRUNET_WIDTH, RESCROSS_WIDTH, NUM_WORKERS = 8, 8, 0
    RUN_LPIPS, MULTI_GPU = False, False

print('device:',torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU',
      '| GPUs:',torch.cuda.device_count())
print('output:',OUTPUT_DIR,'| smoke:',SMOKE_TEST)


In [ ]:
for module, package in [('PIL','Pillow'),('skimage','scikit-image'),('torchmetrics','torchmetrics'),('pandas','pandas'),('matplotlib','matplotlib'),('tqdm','tqdm')]:
    if importlib.util.find_spec(module) is None:
        print('Installing:',package)
        subprocess.check_call([sys.executable,'-m','pip','install','-q',package])
import pandas as pd
import matplotlib.pyplot as plt


In [ ]:
from PIL import Image
if SMOKE_TEST:
    root=WORK_DIR/'div2k_data'
    for part,ids in [('train',range(1,5)),('valid',[801,802,851,852])]:
        folder=root/f'DIV2K_{part}_HR';folder.mkdir(parents=True,exist_ok=True)
        for k in ids:
            p=folder/f'{k:04d}.png'
            if not p.exists():
                rng=np.random.default_rng(k)
                yy,xx=np.mgrid[0:64,0:64]
                rgb=np.stack([np.sin(xx/7+k)*.3+.5,np.cos(yy/5+k)*.3+.5,rng.random((64,64))*.5+.25],axis=-1)
                Image.fromarray(np.uint8(np.clip(rgb,0,1)*255),'RGB').save(p)
    print('SMOKE fake DIV2K data prepared at',root)


In [ ]:
from pathlib import Path
import shutil, urllib.request, zipfile

MANUAL_TRAIN_DIR = None
MANUAL_VALID_DIR = None
DATA_LINKS = {
  'train': 'https://data.vision.ee.ethz.ch/cvl/DIV2K/DIV2K_train_HR.zip',
  'valid': 'https://data.vision.ee.ethz.ch/cvl/DIV2K/DIV2K_valid_HR.zip'
}

def scan_files(roots, wanted):
    dirs, archives = [], []
    for base in roots:
        if not base.exists():
            continue
        for folder, subdirs, files in os.walk(base):
            subdirs[:] = [x for x in subdirs if x not in {'.git', '.cache', '__pycache__'}]
            p = Path(folder)
            if p.name.lower() == wanted.lower() and any(p.glob('*.png')):
                dirs.append(p)
            for name in files:
                if name.lower() == f'{wanted.lower()}.zip':
                    archives.append(p / name)
    return sorted(dirs, key=str), sorted(archives, key=str)

def extract_zip(archive, destination):
    destination.mkdir(parents=True, exist_ok=True)
    print('Rozpakowuję:', archive)
    with zipfile.ZipFile(archive) as z:

        root = destination.resolve()
        for member in z.infolist():
            target = (root / member.filename).resolve()
            if target != root and root not in target.parents:
                raise ValueError('Niebezpieczna ścieżka w ZIP: ' + member.filename)
        z.extractall(root)

DATA_ROOT = WORK_DIR / 'div2k_data'
DATA_ROOT.mkdir(parents=True, exist_ok=True)
for split in ('train', 'valid'):
    wanted = f'DIV2K_{split}_HR'
    manual = MANUAL_TRAIN_DIR if split == 'train' else MANUAL_VALID_DIR
    search_roots = [INPUT_DIR, WORK_DIR / 'div2k_data']
    found_dirs, archives = scan_files(search_roots, wanted)
    if manual is not None:
        selected = Path(manual)
        if not selected.is_dir() or not list(selected.glob('*.png')):
            raise FileNotFoundError('Nieprawidłowy katalog: ' + str(selected))
    elif found_dirs:
        selected = found_dirs[0]
    else:
        if archives:
            extract_zip(archives[0], DATA_ROOT / '_archives')
        elif AUTO_DOWNLOAD_IF_MISSING:
            path = DATA_ROOT / (wanted + '.zip')
            print('Pobieram oficjalne DIV2K:', DATA_LINKS[split])
            urllib.request.urlretrieve(DATA_LINKS[split], path)
            extract_zip(path, DATA_ROOT / '_archives')
            path.unlink(missing_ok=True)
        else:
            raise FileNotFoundError(
                f'Brak {wanted}. Użyj Add Input w Kaggle (katalog PNG lub {wanted}.zip) '
                'albo ustaw AUTO_DOWNLOAD_IF_MISSING=True, jeśli masz Internet.'
            )
        found_dirs, _ = scan_files([DATA_ROOT / '_archives'], wanted)
        if not found_dirs:
            raise FileNotFoundError('Nie znaleziono katalogu z PNG po rozpakowaniu ' + wanted)
        selected = found_dirs[0]
    link = DATA_ROOT / wanted
    if not link.exists() and not link.is_symlink():
        try:
            link.symlink_to(selected.resolve(), target_is_directory=True)
        except OSError:

            link.mkdir()
            for file in selected.glob('*.png'):
                shutil.copy2(file, link / file.name)
    png_count = len(list(link.glob('*.png')))
    print(f'{wanted}: {png_count} PNG ← {selected}')
    expected = 800 if split == 'train' else 100
    if png_count != expected and not SMOKE_TEST:
        raise RuntimeError(f'Niekompletny zbiór: {wanted}: {png_count}, oczekiwano {expected}.')

print('DATA_ROOT=', DATA_ROOT)


In [ ]:
%%writefile sigk_v4/data.py

from __future__ import annotations

import random
from pathlib import Path

import numpy as np
import torch
from PIL import Image, ImageEnhance, ImageOps
from torch.utils.data import Dataset

SIGMAS = (0.01, 0.03)
VALID_IDS = tuple(range(801, 901))
VAL_IDS = tuple(range(801, 851))
TEST_IDS = tuple(range(851, 901))


def find_split_directory(root: str | Path, split: str) -> Path:

    base = Path(root).expanduser()
    actual = "train" if split == "train" else "valid"
    names = (f"DIV2K_{actual}_HR", actual)
    candidates = [base]
    for prefix in (base, base / "DIV2K", base / "raw" / "DIV2K", base / "data" / "DIV2K",
                   base / "p1" / "data" / "raw" / "DIV2K"):
        candidates += [prefix / n for n in names]
    for candidate in candidates:
        if candidate.is_dir() and any(candidate.glob("*.png")):
            return candidate
    raise FileNotFoundError(
        f"Nie znaleziono obrazow {split} w {base}. Oczekiwano DIV2K_{actual}_HR/*.png. "
        "Uruchom python -m sigk_denoising.download albo sprawdz --data-dir."
    )


def list_images(root: str | Path, split: str) -> list[Path]:
    folder = find_split_directory(root, split)
    paths = sorted(folder.glob("*.png"))
    allowed = None
    if split == "val":
        allowed = set(VAL_IDS)
    elif split == "test":
        allowed = set(TEST_IDS)
    if allowed is not None:
        paths = [p for p in paths if p.stem.isdigit() and int(p.stem) in allowed]
    if not paths:
        raise ValueError(f"Brak obrazow PNG dla split={split} w {folder}")
    return paths


def clean_patch(image: Image.Image, size: int, rng: random.Random,
                training: bool, augment: str) -> torch.Tensor:

    image = image.convert("RGB")
    w, h = image.size
    if min(w, h) < size:
        scale = size / min(w, h)
        image = image.resize((round(w * scale), round(h * scale)), Image.Resampling.BICUBIC)
        w, h = image.size
    if training:
        left = rng.randrange(w - size + 1)
        top = rng.randrange(h - size + 1)
    else:
        left, top = (w - size) // 2, (h - size) // 2
    image = image.crop((left, top, left + size, top + size))

    if training and augment in ("geometric", "full"):
        if rng.random() < .5:
            image = ImageOps.mirror(image)
        if rng.random() < .5:
            image = ImageOps.flip(image)
        image = image.rotate(90 * rng.randrange(4))
    if training and augment == "full":

        image = ImageEnhance.Brightness(image).enhance(rng.uniform(.90, 1.10))
        image = ImageEnhance.Contrast(image).enhance(rng.uniform(.90, 1.10))
        image = ImageEnhance.Color(image).enhance(rng.uniform(.90, 1.10))
    rgb = np.array(image, dtype=np.float32).copy() / 255.0
    return torch.from_numpy(rgb).permute(2, 0, 1)


def corrupt(clean: torch.Tensor, sigma: float, seed: int | None = None) -> torch.Tensor:

    if seed is None:
        noise = torch.randn_like(clean)
    else:
        generator = torch.Generator(device="cpu").manual_seed(int(seed))
        noise = torch.randn(clean.shape, dtype=clean.dtype, generator=generator)
    return (clean + noise * sigma).clamp_(0, 1)


class DIV2KDenoising(Dataset):
    def __init__(self, root: str | Path, split: str, size: int = 256,
                 augment: str = "geometric", samples_per_epoch: int = 6400, seed: int = 42):
        if split not in ("train", "val", "test"):
            raise ValueError("split musi byc train/val/test")
        if augment not in ("none", "geometric", "full"):
            raise ValueError("augment: none/geometric/full")
        if size < 16:
            raise ValueError("size >= 16")
        self.paths = list_images(root, split)
        self.split, self.size, self.augment = split, size, augment
        self.samples_per_epoch = samples_per_epoch
        self.seed = seed
        self.epoch = 0

    def set_epoch(self, epoch: int) -> None:
        self.epoch = epoch

    def __len__(self) -> int:
        return self.samples_per_epoch if self.split == "train" else len(self.paths)

    def __getitem__(self, index: int):
        if index < 0 or index >= len(self):
            raise IndexError(index)


        rng = random.Random(f"{self.seed}|{self.epoch if self.split == 'train' else 0}|{index}")
        path = rng.choice(self.paths) if self.split == "train" else self.paths[index]
        with Image.open(path) as image:
            clean = clean_patch(image, self.size, rng, self.split == "train", self.augment)
        sigma = SIGMAS[rng.randrange(len(SIGMAS))] if self.split == "train" else SIGMAS[0]
        noise_seed = rng.getrandbits(63)
        noisy = corrupt(clean, sigma, noise_seed)
        return noisy, clean, torch.tensor(sigma, dtype=torch.float32)


def fixed_noise(clean: torch.Tensor, sigma: float, seed: int, image_index: int, repeat: int = 0):
    return corrupt(clean, sigma, seed + 100_003 * image_index + 10_000_019 * repeat + int(sigma * 1000))

class PairedTrainingDataset(Dataset):

    def __init__(self, root, size=256, samples_per_epoch=4800, seed=42, x4=False):
        self.paths = list_images(root, 'train')
        self.size = size
        self.count = samples_per_epoch
        self.seed = seed
        self.x4 = bool(x4)
        self.epoch = 0

    def set_epoch(self, epoch):
        self.epoch = int(epoch)

    def __len__(self):
        return self.count

    def __getitem__(self, index):
        if not 0 <= index < self.count:
            raise IndexError(index)
        n = len(self.paths)
        image_index = index % n
        cycle = index // n
        crop_id = cycle // 4 if self.x4 else cycle
        rng = random.Random(f'{self.seed}|{self.epoch}|{image_index}|{crop_id}')
        with Image.open(self.paths[image_index]) as img:
            clean = clean_patch(img, self.size, rng, True, 'none')
        if self.x4:
            view = cycle % 4
            if view == 1:
                clean = clean.flip(-1)
            elif view == 2:
                clean = clean.flip(-2)
            elif view == 3:
                clean = clean.rot90(1, dims=(-2, -1))

        sigma = SIGMAS[(cycle + image_index + self.epoch) % 2]
        seed_rng = random.Random(f'noise|{self.seed}|{self.epoch}|{index}')
        noisy = corrupt(clean, sigma, seed_rng.getrandbits(63))
        return noisy, clean, torch.tensor(sigma, dtype=torch.float32)


In [ ]:
%%writefile sigk_v4/legacy.py

import torch
from torch import nn


class DnCNN(nn.Module):


    def __init__(self, depth: int = 17, features: int = 64, conditioned: bool = True):
        super().__init__()
        if depth < 3 or features < 4:
            raise ValueError("depth >= 3, features >= 4")
        self.depth, self.features, self.conditioned = depth, features, conditioned
        in_ch = 4 if conditioned else 3
        layers: list[nn.Module] = [nn.Conv2d(in_ch, features, 3, padding=1), nn.ReLU(inplace=True)]
        for _ in range(depth - 2):
            layers += [nn.Conv2d(features, features, 3, padding=1, bias=False), nn.BatchNorm2d(features), nn.ReLU(inplace=True)]
        layers.append(nn.Conv2d(features, 3, 3, padding=1))
        self.net = nn.Sequential(*layers)

        nn.init.zeros_(self.net[-1].weight)
        nn.init.zeros_(self.net[-1].bias)

    def forward(self, noisy: torch.Tensor, sigma: torch.Tensor | float) -> torch.Tensor:
        if self.conditioned:
            if not torch.is_tensor(sigma):
                sigma = torch.tensor(sigma, device=noisy.device, dtype=noisy.dtype)
            sigma = sigma.to(device=noisy.device, dtype=noisy.dtype)
            if sigma.ndim == 0:
                sigma = sigma.expand(noisy.shape[0])
            if sigma.ndim != 1 or sigma.shape[0] != noisy.shape[0]:
                raise ValueError("sigma musi byc skalarem lub tensorem [batch]")
            sigma_map = (sigma / 0.03).view(-1, 1, 1, 1).expand(-1, -1, noisy.shape[2], noisy.shape[3])
            x = torch.cat((noisy, sigma_map), dim=1)
        else:
            x = noisy
        return noisy - self.net(x) * sigma.view(-1, 1, 1, 1)


In [ ]:
%%writefile sigk_v4/baseline.py

import numpy as np
import torch
from skimage.restoration import denoise_bilateral


def bilateral(noisy: torch.Tensor, sigma: float, sigma_spatial: float = 2.0,
              sigma_color_multiplier: float = 2.0) -> torch.Tensor:

    inp = noisy.detach().cpu().permute(1, 2, 0).numpy().astype(np.float32)
    filtered = denoise_bilateral(inp, sigma_color=sigma_color_multiplier * sigma,
                                 sigma_spatial=sigma_spatial, channel_axis=-1)
    return torch.from_numpy(np.asarray(filtered, dtype=np.float32).copy()).permute(2, 0, 1).clamp(0, 1)


In [ ]:
%%writefile sigk_v4/models.py

from __future__ import annotations
import torch
from torch import nn
from torch.nn import functional as F
from .legacy import DnCNN


def prepare_input(noisy, sigma):
    if not torch.is_tensor(sigma):
        sigma = torch.tensor(sigma, device=noisy.device, dtype=noisy.dtype)
    sigma = sigma.to(device=noisy.device, dtype=noisy.dtype)
    if sigma.ndim == 0:
        sigma = sigma.expand(noisy.size(0))
    if sigma.shape != (noisy.size(0),):
        raise ValueError('Sigma must be scalar or B-sized vector')
    sigmamap = (sigma / 0.03)[:, None, None, None].expand(-1, 1, noisy.shape[-2], noisy.shape[-1])
    return torch.cat([noisy, sigmamap], dim=1), sigma


def norm(ch):
    for groups in (8, 4, 2, 1):
        if ch % groups == 0:
            return nn.GroupNorm(groups, ch)


class ResidualBlock(nn.Module):
    def __init__(self, ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(ch, ch, 3, padding=1), norm(ch), nn.GELU(),
            nn.Conv2d(ch, ch, 3, padding=1), norm(ch))
        self.scale = nn.Parameter(torch.tensor(0.1))

    def forward(self, x):
        return x + self.scale * self.net(x)


class DRUNetLite(nn.Module):

    def __init__(self, width=32):
        super().__init__()
        c1, c2, c3 = width, width*2, width*3
        self.stem = nn.Conv2d(4, c1, 3, padding=1)
        self.e1 = nn.Sequential(ResidualBlock(c1), ResidualBlock(c1))
        self.down1 = nn.Conv2d(c1, c2, 3, stride=2, padding=1)
        self.e2 = nn.Sequential(ResidualBlock(c2), ResidualBlock(c2))
        self.down2 = nn.Conv2d(c2, c3, 3, stride=2, padding=1)
        self.mid = nn.Sequential(ResidualBlock(c3), ResidualBlock(c3), ResidualBlock(c3))
        self.up2 = nn.Conv2d(c3+c2, c2, 3, padding=1)
        self.d2 = nn.Sequential(ResidualBlock(c2), ResidualBlock(c2))
        self.up1 = nn.Conv2d(c2+c1, c1, 3, padding=1)
        self.d1 = nn.Sequential(ResidualBlock(c1), ResidualBlock(c1))
        self.head = nn.Conv2d(c1, 3, 3, padding=1)
        nn.init.zeros_(self.head.weight)
        nn.init.zeros_(self.head.bias)

    def forward(self, noisy, sigma):
        inp, sigma = prepare_input(noisy, sigma)
        x1 = self.e1(self.stem(inp))
        x2 = self.e2(self.down1(x1))
        x3 = self.mid(self.down2(x2))
        x = F.interpolate(x3, size=x2.shape[-2:], mode='bilinear', align_corners=False)
        x = self.d2(self.up2(torch.cat((x, x2), dim=1)))
        x = F.interpolate(x, size=x1.shape[-2:], mode='bilinear', align_corners=False)
        x = self.d1(self.up1(torch.cat((x, x1), dim=1)))
        return noisy - sigma[:, None, None, None] * self.head(x)


class CrossContext(nn.Module):

    def __init__(self, width=64, heads=4):
        super().__init__()
        assert width % heads == 0
        self.local_projection = nn.Conv2d(width, width, 1)
        self.context = nn.Sequential(
            nn.Conv2d(4, width, 5, stride=2, padding=2), nn.GELU(),
            nn.Conv2d(width, width, 3, stride=2, padding=1), nn.GELU(),
            nn.Conv2d(width, width, 3, stride=2, padding=1), nn.GELU())
        self.attn = nn.MultiheadAttention(width, heads, batch_first=True)
        self.post = nn.Conv2d(width, width, 1)
        self.gate = nn.Parameter(torch.tensor(0.1))

    def forward(self, local, original_in):
        b, c, h, w = local.shape
        q_map = F.adaptive_avg_pool2d(self.local_projection(local), (16,16))
        kv_map = F.adaptive_avg_pool2d(self.context(original_in), (8,8))
        q = q_map.flatten(2).transpose(1,2)
        kv = kv_map.flatten(2).transpose(1,2)
        attended, _ = self.attn(q, kv, kv, need_weights=False)
        out = attended.transpose(1,2).reshape(b,c,16,16)
        out = F.interpolate(self.post(out), size=(h,w), mode='bilinear', align_corners=False)
        return local + self.gate * out


class ResCrossDnCNN(nn.Module):

    def __init__(self, width=64):
        super().__init__()
        self.stem = nn.Conv2d(4, width, 3, padding=1)
        self.before = nn.Sequential(*[ResidualBlock(width) for _ in range(4)])
        self.cross = CrossContext(width, heads=4)
        self.after = nn.Sequential(*[ResidualBlock(width) for _ in range(4)])
        self.head = nn.Conv2d(width, 3, 3, padding=1)
        nn.init.zeros_(self.head.weight)
        nn.init.zeros_(self.head.bias)

    def forward(self, noisy, sigma):
        inp, sigma = prepare_input(noisy, sigma)
        x = self.before(self.stem(inp))
        x = self.cross(x, inp)
        x = self.after(x)
        return noisy - sigma[:, None, None, None] * self.head(x)


def make_model(name, config=None):
    config = config or {}
    if name in ('v2', 'v3', 'dncnn'):
        return DnCNN(**(config or {'depth': 17, 'features':64, 'conditioned':True}))
    if name == 'drunet_lite':
        return DRUNetLite(width=config.get('width',32))
    if name == 'rescross':
        return ResCrossDnCNN(width=config.get('width',64))
    raise ValueError('Unrecognized model: ' + str(name))


In [ ]:
%%writefile sigk_v4/training.py

import copy, csv, json, random, time
from pathlib import Path
import numpy as np
import torch
from torch.utils.data import DataLoader
from skimage.metrics import structural_similarity, peak_signal_noise_ratio
from .data import PairedTrainingDataset, DIV2KDenoising, fixed_noise, SIGMAS
from .models import make_model


def optimizer_groups(model):
    decayed, not_decayed = [], []
    for n, p in model.named_parameters():
        if not p.requires_grad: continue
        if p.ndim <= 1 or n.endswith('.bias'):
            not_decayed.append(p)
        else:
            decayed.append(p)
    return [{'params': decayed, 'weight_decay':1e-4},
            {'params':not_decayed, 'weight_decay':0.0}]


def update_ema(ema, model, decay):
    with torch.no_grad():
        src = model.state_dict()
        for key, value in ema.state_dict().items():
            if torch.is_floating_point(value):
                value.lerp_(src[key].detach(), 1.0-decay)
            else:
                value.copy_(src[key])


def build_amp(device, enabled):
    on = bool(enabled and device.type == 'cuda')
    return on, torch.amp.GradScaler('cuda', enabled=on)


def loss_fn(denoised, clean, sigma, ssim_weight):

    pred = denoised.float()
    clean = clean.float()
    sig = sigma.float()[:, None, None, None]
    mse = (((pred-clean) / sig) ** 2).mean()
    if ssim_weight:
        from torchmetrics.functional.image import structural_similarity_index_measure
        val = structural_similarity_index_measure(pred.clamp(0,1), clean, data_range=1.0)
        return mse + ssim_weight*(1.0-val)
    return mse


@torch.no_grad()
def validate(model, root, device, patch_size, seed=1_000_042, allow_small=False):
    model.eval()
    ds = DIV2KDenoising(root, 'val', size=patch_size, augment='none')
    if not allow_small and len(ds) != 50:
        raise ValueError(f'Expected 50 DIV2K val images, got {len(ds)}')

    out = {}
    for sigma in SIGMAS:
        psnr, ssim = [], []
        for i in range(len(ds)):
            clean = ds[i][1]
            noisy = fixed_noise(clean, sigma, seed, i)
            with torch.no_grad():
                y = model(noisy[None].to(device), sigma).clamp(0,1)[0].float().cpu()
            ref = clean.permute(1,2,0).numpy()
            got = y.permute(1,2,0).numpy()
            psnr.append(peak_signal_noise_ratio(ref, got, data_range=1.0))
            ssim.append(structural_similarity(ref, got, channel_axis=-1, data_range=1.0))
        out[f'psnr_{sigma:.2f}'] = float(np.mean(psnr))
        out[f'ssim_{sigma:.2f}'] = float(np.mean(ssim))
    out['psnr'] = float(np.mean([out[f'psnr_{s:.2f}'] for s in SIGMAS]))
    out['ssim'] = float(np.mean([out[f'ssim_{s:.2f}'] for s in SIGMAS]))
    out['selection_score'] = out['psnr'] + 10.0*out['ssim']
    return out


def run_train(name, root, output, settings, augmentation_x4=False, smoke=False):
    out = Path(output); out.mkdir(parents=True,exist_ok=True)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    torch.backends.cudnn.benchmark = bool(device.type == 'cuda')
    seed = int(settings.get('seed', 42))
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if device.type == 'cuda': torch.cuda.manual_seed_all(seed)
    arch = 'drunet_lite' if name.startswith('drunet_lite') else 'rescross'
    model_cfg = {'width':int(settings.get('drunet_width',32) if arch=='drunet_lite' else settings.get('rescross_width',64))}
    model = make_model(arch, model_cfg).to(device)
    ema = copy.deepcopy(model).eval()
    multi = bool(settings.get('multi_gpu',False) and device.type == 'cuda' and torch.cuda.device_count()>1)
    forward_model = torch.nn.DataParallel(model) if multi else model
    lr = float(settings.get('lr',2e-4))
    epochs = int(settings.get('epochs',14))
    per_epoch = int(settings.get('samples_per_epoch',4800))
    batch = int(settings.get('batch',32))
    patch_size = int(settings.get('patch_size',256))
    workers = int(settings.get('workers',4))
    ssim_weight = float(settings.get('ssim_weight',1.0))
    ema_decay = float(settings.get('ema_decay',0.999))
    use_amp, scaler = build_amp(device, settings.get('amp',True))
    opt = torch.optim.AdamW(optimizer_groups(model), lr=lr)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max(epochs,1), eta_min=float(settings.get('lr_min',1e-6)))
    ds = PairedTrainingDataset(root, size=patch_size, samples_per_epoch=per_epoch, seed=seed, x4=augmentation_x4)
    last = out/'last.pt'; best = out/'best.pt'; history = out/'history.csv'
    start, best_score = 1, -1e30
    if last.exists():
        cp = torch.load(last,map_location='cpu',weights_only=True)
        if cp['architecture'] != arch or cp['model_config'] != model_cfg or bool(cp['aug_x4']) != augmentation_x4:
            raise RuntimeError('Existing last.pt has incompatible model or augmentation settings')

        for attr, target in [('epochs',epochs),('samples_per_epoch',per_epoch),('batch',batch),('patch_size',patch_size)]:
            if int(cp['settings'].get(attr,target)) != target:
                raise RuntimeError(f'Cannot resume with changed {attr}; create a new RUN_NAME')
        model.load_state_dict(cp['state_dict']); ema.load_state_dict(cp['ema_state_dict'])
        opt.load_state_dict(cp['optimizer']); scheduler.load_state_dict(cp['scheduler'])
        if cp.get('scaler') and use_amp: scaler.load_state_dict(cp['scaler'])
        best_score = float(cp['best_score'])
        start = int(cp['epoch'])+1
        print(f'RESUME {name}: epoch={start} using {last}',flush=True)
    (out/'settings.json').write_text(json.dumps({'architecture':arch,'model_config':model_cfg,'augmentation_x4':augmentation_x4,
                     'settings':settings,'model_params':sum(p.numel() for p in model.parameters())},indent=2),encoding='utf-8')
    for ep in range(start,epochs+1):
        ds.set_epoch(ep)
        loader = DataLoader(ds,batch_size=batch,shuffle=True,
            num_workers=workers,pin_memory=device.type=='cuda',
            persistent_workers=bool(workers>0),drop_last=True,
            generator=torch.Generator().manual_seed(seed+ep))
        model.train()
        nimages, tot_loss, used_micro, st = 0,0.,batch,time.perf_counter()
        for noisy, clean, sigma in loader:
            noisy = noisy.to(device,non_blocking=True); clean=clean.to(device,non_blocking=True)
            sigma = sigma.to(device,non_blocking=True)


            micro = min(used_micro, len(noisy))
            while True:
                opt.zero_grad(set_to_none=True)
                try:
                    loss_value = 0.
                    for begin in range(0,len(noisy),micro):
                        n = noisy[begin:begin+micro]
                        c = clean[begin:begin+micro]
                        s = sigma[begin:begin+micro]
                        with torch.amp.autocast(device_type='cuda',enabled=use_amp):
                            result = forward_model(n,s)
                        loss = loss_fn(result,c,s,ssim_weight)
                        fraction = len(n)/len(noisy)
                        scaler.scale(loss*fraction).backward()
                        loss_value += float(loss.detach())*fraction
                    scaler.unscale_(opt)
                    torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
                    scaler.step(opt)
                    scaler.update()
                    break
                except torch.cuda.OutOfMemoryError:
                    opt.zero_grad(set_to_none=True)
                    if device.type!='cuda' or micro<=1:
                        raise
                    micro=max(1,micro//2)
                    used_micro=micro
                    torch.cuda.empty_cache()
                    print(f'GPU OOM: lower physical microbatch to {micro}, keep effective batch={batch}',flush=True)
            nimages += len(noisy)
            tot_loss += loss_value*len(noisy)
            update_ema(ema,model,ema_decay)
        scheduler.step()
        if device.type == 'cuda':torch.cuda.synchronize()
        secs=time.perf_counter()-st
        raw=validate(model,root,device,patch_size,allow_small=smoke)
        ema_result=validate(ema,root,device,patch_size,allow_small=smoke)
        chosen = 'raw' if raw['selection_score'] >= ema_result['selection_score'] else 'ema'
        selected = raw if chosen=='raw' else ema_result
        score = selected['selection_score']
        selected_state = model.state_dict() if chosen=='raw' else ema.state_dict()
        checkpoint = dict(epoch=ep,architecture=arch,model_config=model_cfg,aug_x4=augmentation_x4,
              state_dict=copy.deepcopy(model.state_dict()),ema_state_dict=copy.deepcopy(ema.state_dict()),
              optimizer=opt.state_dict(),scheduler=scheduler.state_dict(),scaler=scaler.state_dict(),
              settings=settings,best_score=max(best_score,score),eval_weights=chosen,
              metrics_raw=raw,metrics_ema=ema_result)
        torch.save(checkpoint,last)
        improved=score > best_score
        if improved:
            best_score=score
            checkpoint['eval_state_dict'] = copy.deepcopy(selected_state)
            torch.save(checkpoint,best)
        row=dict(epoch=ep,lr=opt.param_groups[0]['lr'],loss=tot_loss/max(nimages,1),
              train_images=nimages,train_seconds=secs,images_per_second=nimages/max(secs,1e-8),
              microbatch=used_micro,raw_psnr=raw['psnr'],raw_ssim=raw['ssim'],
              ema_psnr=ema_result['psnr'],ema_ssim=ema_result['ssim'],
              chosen=chosen,selected_psnr=selected['psnr'],selected_ssim=selected['ssim'],
              psnr_001=selected['psnr_0.01'],psnr_003=selected['psnr_0.03'],
              ssim_001=selected['ssim_0.01'],ssim_003=selected['ssim_0.03'])
        with history.open('a',newline='') as f:
            dw=csv.DictWriter(f,fieldnames=row.keys())
            if not history.stat().st_size:dw.writeheader()
            dw.writerow(row)
        print(f'[{name}] {ep:02d}/{epochs}  LR={row["lr"]:.2e}  PSNR={row["selected_psnr"]:.3f} '
              f'SSIM={row["selected_ssim"]:.4f}  {chosen}  {row["images_per_second"]:.1f} img/s '
              f'(micro={used_micro}/{batch}) '+('*' if improved else ''),flush=True)
    return best


In [ ]:
%%writefile sigk_v4/benchmark.py

from __future__ import annotations
import csv, json, time, io, zipfile
from collections import defaultdict
from pathlib import Path
import numpy as np
import torch
from skimage.metrics import peak_signal_noise_ratio, structural_similarity
from skimage.restoration import denoise_bilateral
from .data import DIV2KDenoising, fixed_noise, SIGMAS
from .models import make_model


def safe_torch_load(p):
    return torch.load(p,map_location='cpu',weights_only=True)


def find_legacy(input_dir, output_dir, label, explicit=None):
    if explicit:
        p=Path(explicit)
        if not p.is_file():raise FileNotFoundError(f'{label} checkpoint path missing: {p}')
        return p
    root=Path(input_dir)
    if not root.exists():return None

    candidates=[]
    for p in root.rglob('*.pt'):
        if p.name!='best.pt' and label not in p.name.lower():continue
        key=str(p).lower()
        if label in key:candidates.append(p)
    if candidates:
        return sorted(candidates,key=lambda x:len(str(x)))[0]
    for archive_path in root.rglob('*.zip'):
        if not any(w in archive_path.name.lower() for w in ('sigk','dncnn','checkpoint','result')):continue
        with zipfile.ZipFile(archive_path) as z:
            members=[i for i in z.infolist()
                     if i.filename.endswith('best.pt') and (label in i.filename.lower() or label in archive_path.name.lower())
                     and i.file_size < 250_000_000]
            if not members:continue
            member=members[0]
            dest=Path(output_dir)/f'imported_{label}_best.pt'
            dest.parent.mkdir(parents=True,exist_ok=True)
            with z.open(member,'r') as src, dest.open('wb') as out:
                import shutil
                shutil.copyfileobj(src,out)
            return dest
    return None


def find_bilateral(input_dir, work_dir):
    for root in [Path(work_dir),Path(input_dir)]:
        if not root.exists():continue
        results=list(root.rglob('bilateral_val_tuned.json'))
        if results:return results[0]
    for p in Path(input_dir).rglob('*.zip') if Path(input_dir).exists() else []:
        with zipfile.ZipFile(p) as z:
            member=next((i for i in z.infolist() if i.filename.endswith('bilateral_val_tuned.json') and i.file_size<1_000_000),None)
            if member:
                dest=Path(work_dir)/'bilateral_val_tuned.json'
                dest.write_bytes(z.read(member))
                return dest
    return None


def tune_bilateral(data_root, file_path, seed=123, smoke=False):
    ds=DIV2KDenoising(data_root,'val',size=32 if smoke else 256,augment='none')
    if not smoke and len(ds)!=50:raise ValueError('Expected complete 50-image validation')
    candidates=[(a,b) for a in (1.,2.,3.,4.) for b in (.25,.35,.5,.75,1.)]
    best={}


    clean_patches=[ds[i][1] for i in range(len(ds))]
    for sigma in SIGMAS:
        pairs=[(clean.permute(1,2,0).numpy(),
                fixed_noise(clean,sigma,seed,i).permute(1,2,0).numpy())
               for i,clean in enumerate(clean_patches)]
        sums=[]
        for sp,cf in candidates:
            values=[]
            for clean,noisy in pairs:
                filtered=denoise_bilateral(noisy, sigma_color=cf*sigma, sigma_spatial=sp,channel_axis=-1)
                values.append(peak_signal_noise_ratio(clean,filtered,data_range=1.0))
            sums.append((float(np.mean(values)),sp,cf))
        winner=max(sums)
        best[f'sigma_{sigma:.2f}']={'sigma_spatial':winner[1],'sigma_color_multiplier':winner[2]}
        print(f'Bilateral VAL sigma={sigma}: {winner[0]:.3f} dB, spatial={winner[1]}, colorfactor={winner[2]}')
    obj={'protocol':'tuned on DIV2K-valid 0801..0850 ONLY', 'best':best,'seed':seed}
    Path(file_path).write_text(json.dumps(obj,indent=2),encoding='utf8')
    return obj


def load_all_models(paths, device):
    loaded={}
    for label, path in paths.items():
        if path is None or not Path(path).exists():
            print(f'MISSING CHECKPOINT: {label} — skipping; no fabricated scores')
            continue
        checkpoint=safe_torch_load(path)
        if label in ('DnCNN V2','DnCNN V3'):
            model=make_model('v2',checkpoint.get('model_config',{'depth':17,'features':64,'conditioned':True}))
        else:
            model=make_model(checkpoint['architecture'],checkpoint.get('model_config',{}))

        state=checkpoint.get('eval_state_dict',checkpoint.get('state_dict'))
        if state is None:raise KeyError(f'{label} checkpoint lacks eval_state_dict / state_dict')
        model.load_state_dict(state, strict=True)
        model.to(device).eval()
        loaded[label]=model
        print(f'Loaded {label} ({sum(p.numel() for p in model.parameters()):,} parameters) from {path}')
    return loaded


def lpips_alex(device):
    from torchmetrics.image.lpip import LearnedPerceptualImagePatchSimilarity
    try:
        return LearnedPerceptualImagePatchSimilarity(net_type='alex',normalize=True).to(device).eval()
    except Exception as exc:
        raise RuntimeError('Cannot initialize AlexNet LPIPS. Enable Kaggle Internet for weights or Add Input with cached weights.') from exc


def measure(pred, clean, perceptual, device):
    pred=pred.float().clamp(0,1).cpu()
    clean=clean.float().cpu()
    a=pred.permute(1,2,0).numpy(); b=clean.permute(1,2,0).numpy()
    scores={'PSNR':float(peak_signal_noise_ratio(b,a,data_range=1.0)),
            'SSIM':float(structural_similarity(b,a,channel_axis=-1,data_range=1.0))}
    if perceptual is None:scores['LPIPS']=None
    else:
        perceptual.reset()
        with torch.inference_mode():
            scores['LPIPS']=float(perceptual(pred[None].to(device),clean[None].to(device)).item())
    return scores


def benchmark(data_root, out_dir, checkpoint_paths, bilateral_file, seed=123, patch_size=256,
              lpips=True, smoke=False, extra_sigma=None, max_images=None):
    output=Path(out_dir);output.mkdir(parents=True,exist_ok=True)
    ds=DIV2KDenoising(data_root,'test',size=patch_size,augment='none')
    if not smoke and len(ds)!=50:
        raise RuntimeError(f'Test must include all 50 unseen images: got {len(ds)}')
    if not smoke and max_images is not None:
        raise ValueError('Limiting test samples is allowed only in smoke mode')
    total=min(len(ds),max_images or len(ds))
    device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    trained=load_all_models(checkpoint_paths,device)
    perception=lpips_alex(device) if lpips else None
    if bilateral_file is None or not Path(bilateral_file).is_file():
        raise FileNotFoundError('Cannot benchmark without the validation-only bilateral settings')
    tuned=json.loads(Path(bilateral_file).read_text())['best']
    sigmas=list(SIGMAS)
    if extra_sigma is not None:sigmas.append(float(extra_sigma))
    rows, durations = [], defaultdict(lambda:[0.,0])
    panels=[]
    for i in range(total):
        clean=ds[i][1]
        for sigma in sigmas:
            noisy=fixed_noise(clean,sigma,seed,i)
            variants={'Noisy (no processing)':noisy}
            if f'sigma_{sigma:.2f}' in tuned:
                p=tuned[f'sigma_{sigma:.2f}']
                st=time.perf_counter()
                inp=noisy.permute(1,2,0).numpy().astype('float32')
                bilat=denoise_bilateral(inp,sigma_color=p['sigma_color_multiplier']*sigma,
                     sigma_spatial=p['sigma_spatial'],channel_axis=-1)
                variants['Bilateral (skimage)']=torch.from_numpy(np.asarray(bilat,dtype=np.float32).copy()).permute(2,0,1).clamp(0,1)
                durations['Bilateral (skimage)'][0]+=time.perf_counter()-st
                durations['Bilateral (skimage)'][1]+=1
            for label,model in trained.items():
                x=noisy[None].to(device)
                if device.type=='cuda':torch.cuda.synchronize()
                start=time.perf_counter()
                with torch.inference_mode():
                    result=model(x,torch.tensor([sigma],device=device)).clamp(0,1)
                if device.type=='cuda':torch.cuda.synchronize()
                durations[label][0]+=time.perf_counter()-start
                durations[label][1]+=1
                variants[label]=result[0].cpu()
            for method,pred in variants.items():
                scores=measure(pred,clean,perception,device)
                rows.append({'image_id':ds.paths[i].stem,'sigma':sigma,'method':method,**scores})
            if i==0 and sigma in SIGMAS:
                panels.append((sigma,clean,{k:v for k,v in variants.items()}))
        if (i+1)%10==0 or (i+1)==total:
            print(f'Benchmark image {i+1}/{total}',flush=True)
    fields=['image_id','sigma','method','PSNR','SSIM','LPIPS']
    with (output/'per_image.csv').open('w',newline='') as f:
        dw=csv.DictWriter(f,fieldnames=fields);dw.writeheader();dw.writerows(rows)
    groups=defaultdict(list)
    for row in rows:groups[(row['sigma'],row['method'])].append(row)
    summary=[]
    for (sigma,method),values in sorted(groups.items()):
        summary.append({'sigma':sigma,'method':method,'images':len(values),
                 **{m:(float(np.mean([v[m] for v in values])) if values[0][m] is not None else None)
                    for m in ('PSNR','SSIM','LPIPS')}})
    with (output/'results.csv').open('w',newline='') as f:
        dw=csv.DictWriter(f,fieldnames=['sigma','method','images','PSNR','SSIM','LPIPS'])
        dw.writeheader();dw.writerows(summary)
    speeds=[]
    for label,(duration,num) in durations.items():
        speeds.append({'method':label,'images':num,'seconds':duration,'images_per_second':num/max(duration,1e-8),
                       'note':'GPU model forward only (single-image)' if label!='Bilateral (skimage)' else 'CPU bilateral filter'})
    with (output/'inference_speed.csv').open('w',newline='') as f:
        dw=csv.DictWriter(f,fieldnames=['method','images','seconds','images_per_second','note'])
        dw.writeheader();dw.writerows(speeds)
    report={'split':'DIV2K_valid_HR 0851..0900 (internal held-out, NOT official DIV2K test)',
            'images':[p.stem for p in ds.paths[:total]],'seed':seed,
            'noise_definition':'V2/V3 fixed_noise(clean,sigma,123,image_index), clipped to [0,1]',
            'patch':patch_size,'lpips':bool(lpips),'methods':list(trained),
            'missing_checkpoints':[name for name,path in checkpoint_paths.items() if path is None or not Path(path).exists()],
            'bilateral_config':str(bilateral_file),'results':summary,'smoke':bool(smoke)}
    (output/'results.json').write_text(json.dumps(report,indent=2,ensure_ascii=False),encoding='utf8')
    md=['# SIGK 2.2 – benchmark porównawczy (DIV2K)',
        '', 'Test wewnętrzny: 0851–0900, 256×256 central crop, identyczny szum dla metod.',
        'Oficjalny baseline: skimage.restoration.denoise_bilateral (parametry dobrane tylko na VAL).',
        'Wyniki **nie** są wynikami oficjalnego testu DIV2K.',
        '','| σ | Metoda | PSNR ↑ | SSIM ↑ | LPIPS ↓ |','|---|---|---:|---:|---:|']
    for r in summary:
        md.append(f'| {r["sigma"]:.2f} | {r["method"]} | {r["PSNR"]:.4f} | {r["SSIM"]:.5f} | '+
             (f'{r["LPIPS"]:.5f}' if r['LPIPS'] is not None else 'N/A')+' |')
    md+=['','**Protokół:** modele trenowane wyłącznie na DIV2K train 0001..0800, '
          'dobierane na valid 0801..0850, porównanie na held-out 0851..0900.',
        'Dla pary nowej architektury x1 vs x4 liczba próbek/epok/aktualizacji jest jednakowa.',
        'Budżety starszych V2 i V3 różnią się – tych porównań nie interpretuj jako ablacji augmentacji.',
        'Brakujące modele: '+(', '.join(report['missing_checkpoints']) or 'brak')]
    (output/'report.md').write_text('\n'.join(md),encoding='utf8')
    try:
        import matplotlib
        matplotlib.use('Agg')
        import matplotlib.pyplot as plt
        if panels:
            sigma,clean,preds=panels[0]
            labels=['Noisy (no processing)','Bilateral (skimage)','DnCNN V2','DnCNN V3',
                    'DRUNet-Lite','ResCrossAttention','DRUNet-Lite ×4','ResCrossAttention ×4']
            picks=[(k,preds[k]) for k in labels if k in preds]
            fig,axes=plt.subplots(1,len(picks)+1,figsize=(3*(len(picks)+1),3.5))
            for ax,(name,image) in zip(axes,[('Clean',clean)]+picks):
                ax.imshow(image.permute(1,2,0).clamp(0,1).numpy())
                ax.set_title(name,fontsize=8);ax.axis('off')
            fig.tight_layout();fig.savefig(output/'example_comparison.png',dpi=120);plt.close(fig)
        if summary:
            for sigma in SIGMAS:
                vals=[r for r in summary if r['sigma']==sigma]
                fig,ax=plt.subplots(figsize=(10,4));ax.bar([r['method'] for r in vals],[r['PSNR'] for r in vals])
                ax.set_title(f'PSNR — sigma {sigma:.2f}');ax.set_ylabel('dB (higher better)')
                ax.tick_params(axis='x',rotation=55);fig.tight_layout()
                fig.savefig(output/f'psnr_{sigma:.2f}.png',dpi=130);plt.close(fig)
    except Exception as exc: print('Visualization optional warning:',repr(exc))
    return summary


In [ ]:
from sigk_v4.models import make_model
from sigk_v4.data import list_images, PairedTrainingDataset
for arch in ('drunet_lite','rescross'):
    model=make_model(arch,{'width':DRUNET_WIDTH if arch=='drunet_lite' else RESCROSS_WIDTH})
    print(arch, f'{sum(x.numel() for x in model.parameters()):,} parameters')
    with torch.inference_mode():
        x=torch.rand(1,3,min(PATCH_SIZE,32),min(PATCH_SIZE,32))
        assert model(x,torch.tensor([.03])).shape == x.shape
splits={s:list_images(DATA_ROOT,s) for s in ('train','val','test')}
for name,paths in splits.items():print(name,len(paths),paths[0].name,'...',paths[-1].name)
assert len({p.stem for s in splits for p in splits[s]})==sum(len(p) for p in splits.values())
if not SMOKE_TEST: assert tuple(len(splits[s]) for s in ('train','val','test'))==(800,50,50)
for x4 in (False,True):
    ds=PairedTrainingDataset(DATA_ROOT,size=PATCH_SIZE,samples_per_epoch=SAMPLES_PER_EPOCH,x4=x4)
    noisy,clean,sigma=ds[0]
    print('x4' if x4 else 'x1',noisy.shape,'sigma=',float(sigma))


In [ ]:
from sigk_v4.training import run_train
settings = dict(seed=SEED,epochs=EPOCHS,batch=BATCH_SIZE,samples_per_epoch=SAMPLES_PER_EPOCH,
        patch_size=PATCH_SIZE,lr=LEARNING_RATE,lr_min=MIN_LEARNING_RATE,
        ssim_weight=SSIM_WEIGHT,ema_decay=EMA_DECAY,workers=NUM_WORKERS,
        amp=USE_AMP,multi_gpu=MULTI_GPU,drunet_width=DRUNET_WIDTH,rescross_width=RESCROSS_WIDTH)
for name in RUN_VARIANTS:
    path=OUTPUT_DIR/name
    if RUN_TRAINING:
        run_train(name,DATA_ROOT,path,settings,augmentation_x4=name.endswith('_x4'),smoke=SMOKE_TEST)
    else:
        print('Training skipped for',name)


In [ ]:
from sigk_v4.benchmark import find_legacy,find_bilateral,tune_bilateral
if SMOKE_TEST:
    from sigk_v4.models import make_model
    mock_dir=OUTPUT_DIR/'mock_legacy';mock_dir.mkdir(exist_ok=True)
    mock_v2=mock_dir/'v2.pt';mock_v3=mock_dir/'v3.pt'
    fake=make_model('v2',{'depth':4,'features':8,'conditioned':True})
    torch.save({'state_dict':fake.state_dict(),'model_config':{'depth':4,'features':8,'conditioned':True},'epoch':1},mock_v2)
    torch.save({'state_dict':fake.state_dict(),'eval_state_dict':fake.state_dict(),
                'model_config':{'depth':4,'features':8,'conditioned':True},'epoch':1,'eval_weights':'raw'},mock_v3)
    v2,v3=mock_v2,mock_v3
else:
    v2=find_legacy(INPUT_DIR,OUTPUT_DIR,'v2',V2_CHECKPOINT_PATH)
    v3=find_legacy(INPUT_DIR,OUTPUT_DIR,'v3',V3_CHECKPOINT_PATH)
print('DnCNN V2:',v2 or 'MISSING — add V2 best.pt via Kaggle Add Input')
print('DnCNN V3:',v3 or 'MISSING — add V3 best.pt after it completes')

baseline_path=None if FORCE_RETUNE_BASELINE else find_bilateral(INPUT_DIR,OUTPUT_DIR)
if baseline_path is None:
    if RUN_BASELINE_TUNING_IF_MISSING:
        baseline_path=OUTPUT_DIR/'bilateral_val_tuned.json'
        print('Tuning bilateral on VALIDATION only, this may be CPU-intensive...')
        tune_bilateral(DATA_ROOT,baseline_path,seed=EVAL_SEED,smoke=SMOKE_TEST)
    else:
        raise FileNotFoundError('Add bilateral_val_tuned.json in Kaggle Add Input or enable tuning')
print('Baseline config:',baseline_path)


In [ ]:
from sigk_v4.benchmark import benchmark
paths={
 'DnCNN V2':v2,
 'DnCNN V3':v3,
 'DRUNet-Lite':OUTPUT_DIR/'drunet_lite'/'best.pt',
 'ResCrossAttention':OUTPUT_DIR/'rescross'/'best.pt',
 'DRUNet-Lite ×4':OUTPUT_DIR/'drunet_lite_x4'/'best.pt',
 'ResCrossAttention ×4':OUTPUT_DIR/'rescross_x4'/'best.pt',
}
if RUN_EVALUATION:
    summary=benchmark(DATA_ROOT,OUTPUT_DIR/'benchmark',paths,baseline_path,seed=EVAL_SEED,
                      patch_size=PATCH_SIZE,lpips=RUN_LPIPS,smoke=SMOKE_TEST,
                      extra_sigma=EXTRA_SIGMA,max_images=2 if SMOKE_TEST else None)
    comparison=pd.DataFrame(summary)
    display(comparison)


In [ ]:
from IPython.display import Image as IPImage,display,Markdown
for name in RUN_VARIANTS:
    hp=OUTPUT_DIR/name/'history.csv'
    if hp.exists():
        h=pd.read_csv(hp)
        if not h.empty:
            print(name,'best epoch=',int(h.loc[h['selected_psnr'].idxmax(),'epoch']),
                  'peak PSNR=',round(h['selected_psnr'].max(),4),
                  'peak SSIM=',round(h['selected_ssim'].max(),5),
                  'median img/s=',round(h['images_per_second'].median(),1))
if RUN_EVALUATION:
    for fn in ['example_comparison.png','psnr_0.01.png','psnr_0.03.png']:
        p=OUTPUT_DIR/'benchmark'/fn
        if p.exists():display(IPImage(filename=str(p)))
    display(Markdown((OUTPUT_DIR/'benchmark'/'report.md').read_text(encoding='utf8')))


In [ ]:
from zipfile import ZipFile,ZIP_DEFLATED
archive=OUTPUT_DIR/'SIGK_V4_DRUNet_ResCross_x4_Benchmark.zip'
with ZipFile(archive,'w',compression=ZIP_DEFLATED,compresslevel=4) as z:
    for p in OUTPUT_DIR.rglob('*'):
        if not p.is_file() or p==archive or p.name=='last.pt' or p.name.startswith('imported_'):
            continue
        if p.parent.name=='mock_legacy':continue
        z.write(p,p.relative_to(OUTPUT_DIR))
print('READY:',archive,'size MB:',round(archive.stat().st_size/1e6,2))
